<a href="https://colab.research.google.com/github/bobuGG/Agentic_Ai/blob/main/Lab2Agentic_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langgraph langchain-groq -q

In [ ]:
from langchain_groq import ChatGroq
from google.colab import userdata
import os

# The `userdata.get` function can only fetch secrets when running directly in the Colab UI.
# The previous execution resulted in a TimeoutException, meaning the secret 'AgenticAi' was not retrieved.
# If you are not running in the Colab UI, you'll need to provide your API key differently.

GROQ_API_KEY = None
try:
    # Attempt to retrieve from Colab secrets
    GROQ_API_KEY = userdata.get('Agentic')
    print("Attempted to retrieve API key from Colab secrets.")
except Exception as e:
    print(f"Warning: Could not retrieve secret from Colab userdata: {e}")
    print("Please ensure you are running in the Colab UI or provide the API key manually.")

# Fallback to environment variable if not set via userdata
if not GROQ_API_KEY:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
    if GROQ_API_KEY:
        print("API key retrieved from environment variable.")
    else:
        print("API key not found in environment variables either.")
        # As an alternative, you can uncomment the line below and paste your Groq API key directly.
        # GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE" # Replace with your actual key
        # print("WARNING: Using a placeholder for GROQ_API_KEY. Please replace 'YOUR_GROQ_API_KEY_HERE' with your actual key if you uncommented the line above.")



if not GROQ_API_KEY or GROQ_API_KEY == "YOUR_GROQ_API_KEY_HERE":
    print("Error: GROQ_API_KEY is not set or is using a placeholder. Please provide your Groq API key to proceed.")
    llm = None # Ensure llm is not initialized with an invalid key
else:
    llm = ChatGroq(
        api_key=GROQ_API_KEY,
        model="llama-3.1-8b-instant",  # fast & free model
        temperature=0
    )
    print("Groq LLM initialized successfully with the provided API key.")

Attempted to retrieve API key from Colab secrets.
Groq LLM initialized successfully with the provided API key.


In [ ]:
from typing import TypedDict, List

class ChatState(TypedDict):
    messages: List[str]     # conversation history
    name: str               # stored user name
    preferences: str        # stored preferences


In [ ]:
def chatbot(state: ChatState) -> dict:
    user_msg = state['messages'][-1]

    # Retrieve memory
    name = state.get('name', '')
    prefs = state.get('preferences', '')

    # Simple memory extraction rules
    if 'my name is' in user_msg.lower():
        name = user_msg.lower().split('my name is')[-1].strip().title()

    if 'i like' in user_msg.lower():
        prefs = user_msg.lower().split('i like')[-1].strip()

    # Context for LLM
    context = f"The user's name is {name}. They like {prefs}."

    # Generate response
    if llm:
        reply = llm.invoke(context + ' Reply to: ' + user_msg).content
    else:
        reply = "Error: LLM is not initialized. Please ensure your API key is correctly set."

    # Update state
    return {
        'messages': state['messages'] + [reply],
        'name': name,
        'preferences': prefs
    }

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ChatState)

builder.add_node('chatbot', chatbot)
builder.add_edge(START, 'chatbot')
builder.add_edge('chatbot', END)

graph = builder.compile()

In [ ]:
state = {
    'messages': [],
    'name': '',
    'preferences': ''
}

if not llm:
    print("Error: LLM is not initialized. Please set your GROQ_API_KEY before running the chatbot.")
else:
    print("🤖 Chatbot started! Type 'quit' to exit.\n")

    while True:
        user = input("You: ")

        if user.lower() == 'quit':
            print("👋 Goodbye!")
            break

        state['messages'].append(user)

        state = graph.invoke(state)

        print("Bot:", state['messages'][-1])

🤖 Chatbot started! Type 'quit' to exit.

Bot: Nice to meet you, Naveen. What brings you here today?
Bot: Hello Naveen, nice to meet you.
Bot: Hello Naveen, nice to meet you.
